# 05 Decision Engine

## Introduction

The previous modeling and comparison stages selected Random Forest as the primary fraud detection model.

However, a fraud detection system should not stop at producing a binary prediction.

In a real operational setting, model probabilities need to be translated into different levels of intervention depending on transaction risk.

This notebook develops a rule-based Decision Engine that converts the Random Forest fraud probability into operational actions.

The Decision Engine is designed around three principles:

- Fraud detection should prioritize Recall to reduce False Negatives.
- Low-cost authentication methods should be used broadly before escalating transactions to manual review.
- Manual review should be reserved for the highest-risk transactions because analyst capacity is limited.

The final decision framework will translate model probability into risk-based actions such as:

- Approve
- Additional Authentication
- Manual Review

The objective is not to maximize a single machine learning metric, but to create a practical balance between fraud prevention, customer friction, and operational workload.

## Decision Engine Objective

The main objective of this notebook is to answer the following question:

> How should fraud probability scores be translated into operational actions?

The analysis will evaluate multiple probability boundaries and measure their impact on:

- Fraud coverage
- False Negatives
- Customer intervention volume
- Manual review workload
- Operational efficiency

The thresholds defined in this notebook represent business decision boundaries rather than model classification thresholds.


## 5.1 Evaluation Data Setup

The Decision Engine uses the fraud probability produced by the selected Random Forest model.

To evaluate business decision boundaries, the original test labels are reconstructed using the same train-test split configuration used during model development.

Only the target variable and saved Random Forest probability output are required for this stage.

In [3]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Project paths
PROJECT_DIR = Path("/home/rizalkurnia/Projects/DataAnalysis/fraud_detection")
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
MODELS = PROJECT_DIR / "models"

# Reconstruct test labels
y = pd.read_parquet(
    DATA_PROCESSED / "data_optimized.parquet",
    columns=["isFraud"]
)["isFraud"]

_, y_test = train_test_split(
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Load selected model probability
y_proba_rf = joblib.load(
    MODELS / "random_forest_probability.pkl"
)

print(f"Test samples      : {len(y_test):,}")
print(f"Fraud cases       : {y_test.sum():,}")
print(f"Fraud rate        : {y_test.mean():.2%}")
print(f"Probability count : {len(y_proba_rf):,}")

Test samples      : 118,108
Fraud cases       : 4,133
Fraud rate        : 3.50%
Probability count : 118,108


## 5.2 Fraud Risk Score Distribution

Before defining operational risk bands, the distribution of Random Forest fraud probability scores is examined.

The purpose of this analysis is to understand how transactions are distributed across different levels of predicted fraud risk and how actual fraud cases are concentrated within those scores.

This provides the empirical basis for defining business decision boundaries rather than relying directly on the classification threshold identified during model comparison.

In [4]:
import numpy as np

risk_distribution = pd.DataFrame({
    "fraud_probability": y_proba_rf,
    "actual_fraud": y_test.to_numpy()
})

bins = np.arange(0.0, 1.01, 0.10)

risk_distribution["risk_bin"] = pd.cut(
    risk_distribution["fraud_probability"],
    bins=bins,
    include_lowest=True
)

risk_summary = (
    risk_distribution
    .groupby("risk_bin", observed=False)
    .agg(
        Transactions=("actual_fraud", "size"),
        Fraud_Cases=("actual_fraud", "sum")
    )
    .reset_index()
)

risk_summary["Fraud_Rate"] = (
    risk_summary["Fraud_Cases"]
    / risk_summary["Transactions"]
)

risk_summary

,risk_bin,Transactions,Fraud_Cases,Fraud_Rate
0,"(-0.001, 0.1]",111610,937,0.008395
1,"(0.1, 0.2]",2945,515,0.174873
2,"(0.2, 0.3]",904,390,0.431416
3,"(0.3, 0.4]",486,310,0.637860
4,"(0.4, 0.5]",394,307,0.779188
5,"(0.5, 0.6]",296,248,0.837838
6,"(0.6, 0.7]",289,264,0.913495
7,"(0.7, 0.8]",349,332,0.951289
8,"(0.8, 0.9]",337,332,0.985163
9,"(0.9, 1.0]",498,498,1.000000


## 5.3 Candidate Risk-Band Scenarios

The next step evaluates several candidate probability boundaries for translating fraud risk scores into operational actions.

Each scenario contains three actions:

- **Approve** — low-risk transactions with no additional intervention
- **Additional Authentication** — medium-risk transactions requiring a low-cost verification step
- **Manual Review** — high-risk transactions requiring analyst intervention

The scenarios are evaluated based on transaction volume, fraud concentration, and fraud coverage within each action band.

No final decision boundaries are selected at this stage.

In [5]:
candidate_scenarios = {
    "Scenario A": {
        "auth_threshold": 0.20,
        "review_threshold": 0.40
    },
    "Scenario B": {
        "auth_threshold": 0.25,
        "review_threshold": 0.40
    },
    "Scenario C": {
        "auth_threshold": 0.20,
        "review_threshold": 0.50
    }
}

In [6]:
scenario_results = []

for scenario_name, thresholds in candidate_scenarios.items():

    auth_threshold = thresholds["auth_threshold"]
    review_threshold = thresholds["review_threshold"]

    actions = pd.cut(
        y_proba_rf,
        bins=[
            -float("inf"),
            auth_threshold,
            review_threshold,
            float("inf")
        ],
        labels=[
            "Approve",
            "Additional Authentication",
            "Manual Review"
        ],
        right=False
    )

    scenario_df = pd.DataFrame({
        "Action": actions,
        "Actual_Fraud": y_test.to_numpy()
    })

    summary = (
        scenario_df
        .groupby("Action", observed=False)
        .agg(
            Transactions=("Actual_Fraud", "size"),
            Fraud_Cases=("Actual_Fraud", "sum")
        )
        .reset_index()
    )

    summary["Fraud_Rate"] = (
        summary["Fraud_Cases"] /
        summary["Transactions"]
    )

    summary["Scenario"] = scenario_name

    scenario_results.append(summary)

risk_band_comparison = pd.concat(
    scenario_results,
    ignore_index=True
)

risk_band_comparison[
    [
        "Scenario",
        "Action",
        "Transactions",
        "Fraud_Cases",
        "Fraud_Rate"
    ]
]

,Scenario,Action,Transactions,Fraud_Cases,Fraud_Rate
0,Scenario A,Approve,114490,1430,0.012490
1,Scenario A,Additional Authentication,1435,710,0.494774
2,Scenario A,Manual Review,2183,1993,0.912964
3,Scenario B,Approve,115054,1641,0.014263
4,Scenario B,Additional Authentication,871,499,0.572905
5,Scenario B,Manual Review,2183,1993,0.912964
6,Scenario C,Approve,114490,1430,0.012490
7,Scenario C,Additional Authentication,1832,1017,0.555131
8,Scenario C,Manual Review,1786,1686,0.944009


## 5.4 Decision Engine Performance Summary

The selected candidate Decision Engine configuration is evaluated based on routing quality and operational workload.

The current risk-band structure is:

- **Approve**: probability < 0.20
- **Additional Authentication**: 0.20 ≤ probability < 0.50
- **Manual Review**: probability ≥ 0.50

The evaluation focuses only on outcomes that can be directly measured from the available data.

Because the dataset does not contain real-world intervention outcomes, this analysis does not estimate how many fraudulent transactions would actually be prevented by authentication or manual review.

In [7]:
decision_actions = pd.cut(
    y_proba_rf,
    bins=[
        -float("inf"),
        0.20,
        0.50,
        float("inf")
    ],
    labels=[
        "Approve",
        "Additional Authentication",
        "Manual Review"
    ],
    right=False
)

decision_engine_df = pd.DataFrame({
    "Action": decision_actions,
    "Actual_Fraud": y_test.to_numpy()
})

decision_summary = (
    decision_engine_df
    .groupby("Action", observed=False)
    .agg(
        Transactions=("Actual_Fraud", "size"),
        Fraud_Cases=("Actual_Fraud", "sum")
    )
    .reset_index()
)

decision_summary["Transaction_Share"] = (
    decision_summary["Transactions"]
    / len(decision_engine_df)
)

decision_summary["Fraud_Rate"] = (
    decision_summary["Fraud_Cases"]
    / decision_summary["Transactions"]
)

decision_summary["Fraud_Coverage"] = (
    decision_summary["Fraud_Cases"]
    / y_test.sum()
)

decision_summary

,Action,Transactions,Fraud_Cases,Transaction_Share,Fraud_Rate,Fraud_Coverage
0,Approve,114490,1430,0.969367,0.012490,0.345996
1,Additional Authentication,1832,1017,0.015511,0.555131,0.246068
2,Manual Review,1786,1686,0.015122,0.944009,0.407936


## 5.5 Decision Engine Routing Summary

The final Decision Engine routes transactions into three operational actions based on the Random Forest fraud probability score:

- **Approve**: probability < 0.20
- **Additional Authentication**: 0.20 ≤ probability < 0.50
- **Manual Review**: probability ≥ 0.50

The purpose of this section is to summarize how transaction volume and fraud cases are distributed across these actions.

In [8]:
decision_summary_display = decision_summary.copy()

decision_summary_display["Transaction_Share"] = (
    decision_summary_display["Transaction_Share"] * 100
)

decision_summary_display["Fraud_Rate"] = (
    decision_summary_display["Fraud_Rate"] * 100
)

decision_summary_display["Fraud_Coverage"] = (
    decision_summary_display["Fraud_Coverage"] * 100
)

decision_summary_display = decision_summary_display.rename(
    columns={
        "Transaction_Share": "Transaction Share (%)",
        "Fraud_Rate": "Fraud Rate (%)",
        "Fraud_Coverage": "Fraud Coverage (%)"
    }
)

decision_summary_display.round(2)

,Action,Transactions,Fraud_Cases,Transaction Share (%),Fraud Rate (%),Fraud Coverage (%)
0,Approve,114490,1430,96.94,1.25,34.60
1,Additional Authentication,1832,1017,1.55,55.51,24.61
2,Manual Review,1786,1686,1.51,94.40,40.79


## 5.6 Decision Engine Interpretation

The final Decision Engine creates a clear separation between low-, medium-, and high-risk transactions.

Approximately 96.94% of transactions are approved without additional intervention, preserving a low-friction customer experience.

Additional Authentication is applied to only 1.55% of transactions, yet this segment contains 24.61% of all fraud cases and has a fraud rate of 55.51%.

Manual Review is reserved for only 1.51% of transactions. This segment is highly concentrated with fraud, with a fraud rate of 94.40%, while covering 40.79% of all fraud cases.

Overall, the two intervention bands affect approximately 3.06% of transactions while routing 65.40% of all fraud cases toward additional controls.

This configuration reflects the project's business priorities:

- maintain low friction for the majority of legitimate transactions,
- use lower-cost authentication for medium-risk transactions,
- reserve manual review for highly concentrated fraud risk,
- accept a controlled level of residual fraud in the low-risk segment.

The Decision Engine therefore provides a practical operational layer on top of the Random Forest fraud probability score.

## 5.7 Conclusion

This notebook developed the operational Decision Engine that translates Random Forest fraud probability scores into risk-based transaction actions.

The final decision structure is:

- **Approve**: probability < 0.20
- **Additional Authentication**: 0.20 ≤ probability < 0.50
- **Manual Review**: probability ≥ 0.50

The resulting routing behavior shows that:

- 96.94% of transactions are approved without additional intervention.
- 1.55% of transactions are routed to Additional Authentication.
- 1.51% of transactions are routed to Manual Review.
- The two intervention bands together cover 65.40% of all fraud cases while affecting only 3.06% of total transactions.
- The Manual Review segment is highly concentrated with fraud, with a fraud rate of 94.40%.

These results demonstrate that the model probability can be translated into a practical operational framework without relying on a single binary classification threshold.

The Decision Engine intentionally balances three competing objectives:

- reducing fraud that passes through the system,
- minimizing unnecessary customer friction,
- limiting expensive manual review workload.

The analysis is limited to routing quality and operational workload because the dataset does not contain real-world outcomes for authentication or manual review interventions.

Therefore, the Decision Engine should be interpreted as a risk-routing framework rather than a direct estimate of fraud prevented.

The next stage of the project will consolidate the modeling results, Decision Engine findings, limitations, and business recommendations into the final report.